# DEH 30-Day PySpark Challenge
## Day 4 — DataFrame Reader: CSV Options & Reading Modes

**Instructions:**
1. Click `File → Save a copy in Drive` before editing
2. Run each cell in order using `Shift + Enter`
3. Complete the assignment cells at the bottom

---

In [ ]:
!pip install pyspark --quiet

In [ ]:
import os
os.environ['AWS_ACCESS_KEY_ID']     = 'your-access-key-here'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'your-secret-key-here'
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'
BUCKET = 'deh-pyspark-challenge-your-account-id'
print('Credentials set.')

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

spark = (
    SparkSession.builder
    .appName("DEH-PySpark-Challenge")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.4.0,com.amazonaws:aws-java-sdk-bundle:1.12.367")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)
print(f"Spark version: {spark.version}")

## Step 4 — Three ways to pass reader options

In [ ]:
# Style 1 — keyword arguments
df1 = spark.read.csv(f"s3a://{BUCKET}/data/orders.csv", header=True, inferSchema=True)

# Style 2 — chained .option() calls (preferred in production)
df2 = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(f"s3a://{BUCKET}/data/orders.csv")

print(f"Both styles produce the same result: {df1.count() == df2.count()}")

## Step 5 — Reading with explicit schema and options

In [ ]:
orders_schema = StructType([
    StructField("order_id",       StringType(),  True),
    StructField("customer_id",    StringType(),  True),
    StructField("product_id",     StringType(),  True),
    StructField("order_date",     DateType(),    True),
    StructField("quantity",       IntegerType(), True),
    StructField("unit_price",     DoubleType(),  True),
    StructField("discount_pct",   IntegerType(), True),
    StructField("status",         StringType(),  True),
    StructField("payment_method", StringType(),  True),
    StructField("region",         StringType(),  True)
])

orders_df = spark.read \
    .option("header", "true") \
    .option("dateFormat", "yyyy-MM-dd") \
    .option("nullValue", "N/A") \
    .option("mode", "PERMISSIVE") \
    .schema(orders_schema) \
    .csv(f"s3a://{BUCKET}/data/orders.csv")

orders_df.printSchema()
orders_df.show(3)

## Step 6 — Reading Modes

In [ ]:
# PERMISSIVE with _corrupt_record column
schema_with_corrupt = StructType([
    StructField("order_id",        StringType(),  True),
    StructField("customer_id",     StringType(),  True),
    StructField("product_id",      StringType(),  True),
    StructField("order_date",      DateType(),    True),
    StructField("quantity",        IntegerType(), True),
    StructField("unit_price",      DoubleType(),  True),
    StructField("discount_pct",    IntegerType(), True),
    StructField("status",          StringType(),  True),
    StructField("payment_method",  StringType(),  True),
    StructField("region",          StringType(),  True),
    StructField("_corrupt_record", StringType(),  True)
])

df_permissive = spark.read \
    .option("header", "true") \
    .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .schema(schema_with_corrupt) \
    .csv(f"s3a://{BUCKET}/data/orders.csv")

# cache() is required before filtering on _corrupt_record
# _corrupt_record is generated during parsing — Spark needs the DataFrame
# materialized in memory before it can filter on this internal column
df_permissive.cache()

print(f"Total rows : {df_permissive.count()}")
print(f"Bad rows   : {df_permissive.filter(df_permissive._corrupt_record.isNotNull()).count()}")
# Result: 0 bad rows — orders.csv is clean data

In [ ]:
# DROPMALFORMED — silently drops bad rows
df_drop = spark.read \
    .option("header", "true") \
    .option("mode", "DROPMALFORMED") \
    .schema(orders_schema) \
    .csv(f"s3a://{BUCKET}/data/orders.csv")

print(f"Rows with DROPMALFORMED: {df_drop.count()}")
# Result: 100 — same as orders.csv is clean, nothing dropped

In [ ]:
# FAILFAST — throws exception on first bad row
df_strict = spark.read \
    .option("header", "true") \
    .option("mode", "FAILFAST") \
    .schema(orders_schema) \
    .csv(f"s3a://{BUCKET}/data/orders.csv")

print(f"Rows with FAILFAST: {df_strict.count()}")
# Result: 100 — no exception because orders.csv is clean

## Step 7 — Demonstrating Modes on Dirty Data

`orders_dirty.csv` has 15 rows with 4 intentionally bad records:
- **O0004** — too few columns
- **O0005** — quantity is 'three' instead of a number
- **O0007** — order_date is 'NOT_A_DATE'
- **O0012** — too many columns
- **O0014** — unit_price is 'NOT_A_NUMBER'

Watch how each mode handles these differently.

In [ ]:
# PERMISSIVE on dirty data — keeps all rows, nulls out bad fields
df_dirty_permissive = spark.read \
    .option("header", "true") \
    .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .schema(schema_with_corrupt) \
    .csv(f"s3a://{BUCKET}/data/orders_dirty.csv")

df_dirty_permissive.cache()

total = df_dirty_permissive.count()
bad   = df_dirty_permissive.filter(df_dirty_permissive._corrupt_record.isNotNull()).count()

print(f"Total rows : {total}")
print(f"Bad rows   : {bad}")
print(f"Good rows  : {total - bad}")
print()
print("Bad rows content:")
df_dirty_permissive.filter(df_dirty_permissive._corrupt_record.isNotNull()) \
    .select("_corrupt_record") \
    .show(truncate=False)

In [ ]:
# DROPMALFORMED on dirty data — silently removes bad rows
df_dirty_drop = spark.read \
    .option("header", "true") \
    .option("mode", "DROPMALFORMED") \
    .schema(orders_schema) \
    .csv(f"s3a://{BUCKET}/data/orders_dirty.csv")

print(f"Rows with DROPMALFORMED: {df_dirty_drop.count()}")
print("Bad rows were silently removed — no indication of how many were dropped")

In [ ]:
# FAILFAST on dirty data — throws exception on first bad row
# Wrap in try/except so the notebook doesn't stop
try:
    df_dirty_strict = spark.read \
        .option("header", "true") \
        .option("mode", "FAILFAST") \
        .schema(orders_schema) \
        .csv(f"s3a://{BUCKET}/data/orders_dirty.csv")
    df_dirty_strict.count()
except Exception as e:
    print(f"FAILFAST threw an exception as expected:")
    print(str(e)[:300])

---
## Assignment — Day 4

---

### Task 1
Read `customers.csv` from S3 using the chained `.option()` style with `header=true` and an explicit schema.  
Columns: `customer_id` (String), `first_name` (String), `last_name` (String), `email` (String), `city` (String), `state` (String), `country` (String), `signup_date` (Date), `segment` (String).  
Show the first 5 rows.

In [ ]:
# Task 1 — Write your code here


### Task 2
Read `orders.csv` using PERMISSIVE mode with a `_corrupt_record` column in the schema.  
Print the total row count and the count of rows where `_corrupt_record` is not null.

In [ ]:
# Task 2 — Write your code here


### Task 3
Read `orders.csv` using DROPMALFORMED mode. Read it again using PERMISSIVE mode.  
Compare the row counts. Are they the same? Why?

In [ ]:
# Task 3 — Write your code here


---
*DEH 30-Day PySpark Challenge · Data Engineering Hub · RADE Program*